# Unit 4, Lecture 2: Routing and conditional handoff

Last lecture, fan-out sent the ticket to **every** reviewer. Today the graph
makes a **decision**: look at the work, and send it to exactly **one** specialist.

- **Fan-out** (L1): one to many, all run.
- **Routing** (today): one of many, chosen. Only that one runs.

Both are how a graph directs work. Real systems use both: route to the right
team, then within a team fan out the independent checks.

Runs on the real `agent-framework` switch-case edges, **offline**, because the
classifier and handlers are plain functions.

## Decide in one place, act in another

One `classify` node decides the queue and attaches it to the ticket. Every edge
after it just reads that decision. The choice is made in exactly one place.

In [ ]:
from cse476.routing import classify, Routed, QUEUES

print("the queues this router knows:", QUEUES)
print("classify attaches a decision to the ticket, then the edges route on it")

## The routing table, as a switch-case

`Case(condition, target)` is one branch: if the queue is billing, go to the
billing handler. `Default(target)` is the else. Read the whole thing as an
if/elif/elif/else chain.

In [ ]:
from cse476.routing import build_router

wf = build_router()
print("router built:", type(wf).__name__)
print()
print("routing table:")
print("  billing?    -> billing_handler")
print("  technical?  -> technical_handler")
print("  account?    -> account_handler")
print("  otherwise   -> general_handler   (the Default)")

## Run it: exactly one handler per ticket

In [ ]:
from cse476.routing import run_router

print(await run_router("I was charged twice, need a refund"))
print(await run_router("the app crashes with an error"))
print(await run_router("I am locked out, forgot my password"))
print(await run_router("hello, just saying hi"))   # matches nothing -> default

Billing went to billing, technical to technical, account to account, and the
greeting that matched no specific case fell to the **default**. Exactly one
handler produced output each time. That is routing: the graph chose one path.

## Trap one: first match wins

Cases are checked **in order**, and the first condition that returns true wins,
exactly like an if/elif chain. A ticket that could match two cases goes to
whichever comes **first** in your list.

In [ ]:
# this ticket mentions BOTH a charge and an error
out = await run_router("I was charged for a plan that gives an error")
print(out)
print()
print("It went to BILLING, because the billing Case is listed before technical.")
print("Order your cases most-specific first. Ordering is tie-break logic.")

## Trap two: the missing default (the dangerous one)

Here is a real subtlety. The `switch_case_edge_group` you just used **requires**
a default; the framework refuses to build without one, because dropping work is
so dangerous. That is the framework protecting you.

But if you hand-roll routing with plain conditional `add_edge` calls, there is
**no** such protection. Unmatched work silently drops: no error, no output, gone.
In a support system that is a customer whose ticket was silently lost, which is
worse than a crash because a crash you notice.


In [ ]:
# First, prove the switch-case REFUSES to drop work: it requires a default.
from agent_framework import WorkflowBuilder, WorkflowContext, executor, Case
from cse476.routing import classify, billing_handler, technical_handler

try:
    (WorkflowBuilder(start_executor=classify)
        .add_switch_case_edge_group(classify, [
            Case(condition=lambda r: r.queue == "billing", target=billing_handler),
            # no Default on purpose
        ]).build())
except ValueError as e:
    print("switch-case refused to build:", e)

# Now the DANGER: a hand-rolled router with plain conditional edges has NO such
# guard. Unmatched work drops silently.
leaky = (WorkflowBuilder(start_executor=classify)
    .add_edge(classify, billing_handler, condition=lambda r: r.queue == "billing")
    .build())

result = await leaky.run("hello, just saying hi")   # matches nothing
print("hand-rolled router, unmatched ticket outputs:", result.get_outputs())
print()
print("Empty. The ticket vanished with no error. The switch-case would not let")
print("this happen; the hand-rolled conditional edges did. THAT is why you use")
print("switch-case with a Default, and do not hand-roll routing.")


Two lessons in one. The framework's `switch_case_edge_group` **protects** you by
requiring a `Default`, so a ticket can never fall through. But the moment you
hand-roll routing with bare conditional edges, that protection is gone and work
drops silently. **Use switch-case with a default; do not reinvent routing by hand.**


## The mapping, and fan-out vs route

In [ ]:
from cse476.routing import ROUTING_MAP, fan_out_vs_route

for concept, tie in ROUTING_MAP.items():
    print(f"{concept:26} ->  {tie}")
print()
for k, v in fan_out_vs_route().items():
    print(f"{k:20}: {v}")

## Your turn

**1. Add a fifth queue.** For example `sales`, with its own `Case` and handler.
You add one `Case` and one handler, and the routing table stays readable.

**2. Break it on purpose.** Remove the `Default` and send a ticket that matches
no case. Watch it vanish with no output. Then put the `Default` back. You have
now felt the silent drop; you will never forget the default again.

**3. Route then fan.** Design a graph that first routes a ticket to a team, then
inside the technical team fans out to two parallel checks. Sketch it. That
nesting, route then fan, is how real systems combine the two shapes.

In [ ]:
# your work here
